# Source-release notes

MFM VAE training, loss tracking, reconstruction, and latent-space analysis.

Run this notebook with its containing directory as the kernel working directory. Experimental inputs and saved outputs are not included in this source release. See the [workflow guide](../docs/WORKFLOW.md), [data requirements](../docs/DATA.md), and [reproducibility notes](../docs/REPRODUCIBILITY.md) before execution.

Scientific settings are preserved. Packaging changes are recorded in [PREPARATION.md](../docs/PREPARATION.md). Original cell numbers in that record do not include this added introductory cell.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import os
import random
import cv2
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import math
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from skimage.metrics import structural_similarity as ssim

In [ ]:
class MFMDataset(Dataset):
    def __init__(self, csv_file, img_dir, img_size=(64, 64)):
        self.labels_df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.img_size = img_size
        # Start indexing from 1, as per requirement
        self.indices = self.labels_df.index + 1  # Assuming segments are named from 1 onward

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        # Get the segment name based on the adjusted index
        segment_number = self.indices[idx]
        img_name = os.path.join(self.img_dir, f"mfm_segment_{segment_number}.png")
        
        try:
            image = cv2.imread(img_name, cv2.IMREAD_COLOR)
            if image is None:
                raise ValueError(f"Image at {img_name} could not be loaded.")
        except Exception as e:
            print(f"Error loading image at {img_name}: {e}")
            return None  # Skip this image in the batch

        # Rest of the image processing remains the same as before
        # Resize while keeping the aspect ratio
        h, w, _ = image.shape
        aspect_ratio = w / h
        if w > h:
            new_w = self.img_size[1]
            new_h = int(new_w / aspect_ratio)
        else:
            new_h = self.img_size[0]
            new_w = int(new_h * aspect_ratio)
        resized_image = cv2.resize(image, (new_w, new_h))

        # Create a new image with the target size and paste the resized image at the center
        padded_image = np.zeros((self.img_size[0], self.img_size[1], 3), dtype=np.uint8)
        y_offset = (self.img_size[0] - new_h) // 2
        x_offset = (self.img_size[1] - new_w) // 2
        padded_image[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized_image

        # Convert BGR to RGB
        image = cv2.cvtColor(padded_image, cv2.COLOR_BGR2RGB)

        # Normalize and convert to tensor
        image = torch.from_numpy(image).float() / 255.0
        image = image.permute(2, 0, 1)

        # Get labels
        direction_binary = torch.tensor(self.labels_df.iloc[idx, 1], dtype=torch.float32)
        strength = torch.tensor(self.labels_df.iloc[idx, 2], dtype=torch.float32)

        return image, direction_binary, strength, segment_number  # Return segment number for correct indexing


In [ ]:
img_dir = "mfm_segments"  # Directory with images
csv_file = "{}/mfm_segments_with_arrows/magnetic_moments.csv".format(os.getcwd())                          

# Define dataset and dataloaders
img_size = (64, 64)
dataset = MFMDataset(csv_file=csv_file, img_dir=img_dir, img_size=img_size)

# Create training and validation split with consistent segment numbering
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Split by segment order
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

def collate_fn(batch):
    # Filter out None values
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)


In [ ]:
# VAE Model Definition
class VAE(nn.Module):
    def __init__(self, input_dim=64 * 64 * 3, hidden_dim=400, latent_dim=60):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mean = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2_mean(h), self.fc2_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 64 * 64 * 3))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    BCE = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 64 * 64 * 3), reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

In [ ]:
# def train_vae(model, train_loader, optimizer, num_epochs=100):
#     model.train()
#     for epoch in range(num_epochs):
#         total_loss = 0
#         for images, _, _, _ in train_loader:  # Ignore additional items
#             images = images.view(-1, 64 * 64 * 3)  # Flatten the images
#             optimizer.zero_grad()
#             recon_images, mu, logvar = model(images)
#             loss = vae_loss(recon_images, images, mu, logvar)
#             loss.backward()
#             optimizer.step()
#             total_loss += loss.item()
#         print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader.dataset):.4f}")

def train_vae(model, train_loader, optimizer, num_epochs=100):
    model.train()
    total_losses = []  # To track total loss for plotting
    recon_losses = []  # To track reconstruction loss
    kld_losses = []    # To track KL divergence loss
    
    for epoch in range(num_epochs):
        total_loss = 0
        total_recon_loss = 0
        total_kld_loss = 0

        for images, _, _, _ in train_loader:  # Ignore additional items
            images = images.view(-1, 64 * 64 * 3)  # Flatten the images
            optimizer.zero_grad()
            
            # Forward pass
            recon_images, mu, logvar = model(images)
            
            # Compute losses
            BCE = nn.functional.binary_cross_entropy(recon_images, images, reduction='sum')  # Reconstruction loss
            KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())  # KL divergence
            
            loss = BCE + KLD  # Total loss
            loss.backward()
            optimizer.step()

            # Track losses
            total_loss += loss.item()
            total_recon_loss += BCE.item()
            total_kld_loss += KLD.item()

        # Average losses for this epoch
        avg_loss = total_loss / len(train_loader.dataset)
        avg_recon_loss = total_recon_loss / len(train_loader.dataset)
        avg_kld_loss = total_kld_loss / len(train_loader.dataset)

        total_losses.append(avg_loss)
        recon_losses.append(avg_recon_loss)
        kld_losses.append(avg_kld_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}], Total Loss: {avg_loss:.4f}, Recon Loss: {avg_recon_loss:.4f}, KLD Loss: {avg_kld_loss:.4f}")
    
    return total_losses, recon_losses, kld_losses


In [ ]:
# Define output directory for saving synthetic images
output_reconstructed_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
os.makedirs(output_reconstructed_dir, exist_ok=True)

In [ ]:
def test_vae(model, test_loader, original_indices):
    # Function implementation remains the same
    model.eval()
    batch_count = 0  # Counter to track batches

    with torch.no_grad():
        for batch_idx, (images, _, _, segment_numbers) in enumerate(test_loader):  # Adjust to unpack four values
            print(f"Processing batch {batch_idx + 1}...")

            # Flatten images to match VAE input
            images = images.view(-1, 64 * 64 * 3)
            recon_images, _, _ = model(images)
            recon_images = recon_images.view(-1, 3, 64, 64)  # Reshape to image dimensions

            # Save each reconstructed image with the correct index as in the original dataset
            for i in range(recon_images.size(0)):
                # Retrieve the correct segment index from `segment_numbers`
                segment_index = segment_numbers[i].item()  # Convert to a standard Python int for filename use
                print(f"Saving image with segment index: {segment_index}")

                # Convert tensor to numpy, denormalize to [0, 255], and convert to uint8
                reconstructed_img = recon_images[i].permute(1, 2, 0).cpu().numpy() * 255.0
                reconstructed_img = reconstructed_img.astype(np.uint8)

                # Convert RGB to BGR for OpenCV compatibility
                reconstructed_img_bgr = cv2.cvtColor(reconstructed_img, cv2.COLOR_RGB2BGR)

                # Save the reconstructed image using the segment index
                save_path = os.path.join(output_reconstructed_dir, f"synthetic_mfm_segment_{segment_index}.png")
                cv2.imwrite(save_path, reconstructed_img_bgr)

            # Plot example original and reconstructed images for verification
            fig, axes = plt.subplots(2, min(8, recon_images.size(0)), figsize=(15, 4))
            fig.suptitle("Top: Original Segments | Bottom: Reconstructed Segments", fontsize=14)
            for i in range(min(8, recon_images.size(0))):
                segment_index = segment_numbers[i].item()
                
                # Plot original image with segment number in the title
                axes[0, i].imshow(images[i].view(3, 64, 64).permute(1, 2, 0).cpu().numpy())
                axes[0, i].set_title(f"Original Segment {segment_index}")
                axes[0, i].axis("off")
                
                # Plot reconstructed image with segment number in the title
                axes[1, i].imshow(recon_images[i].permute(1, 2, 0).cpu().numpy())
                axes[1, i].set_title(f"Reconstructed Segment {segment_index}")
                axes[1, i].axis("off")
            
            plt.tight_layout()
            plt.subplots_adjust(top=0.85)
            plt.show()

            # Process only one batch for verification
            batch_count += 1
            if batch_count >= 1:
                break

In [ ]:
# Get indices from the DataLoader subset
original_indices = val_loader.dataset.indices  # Retrieve indices directly

# Initialize the model and optimizer
model = VAE(input_dim=64 * 64 * 3, hidden_dim=400, latent_dim=30)
optimizer = optim.Adam(model.parameters(), lr=0.005)

# Train the model and generate synthetic images for the test set
train_vae(model, train_loader, optimizer, num_epochs=100)
test_vae(model, val_loader, original_indices)

In [ ]:
# Train the model and log losses
#total_losses, recon_losses, kld_losses = train_vae(model, train_loader, optimizer, num_epochs=50)


plt.figure(figsize=(10, 5))
plt.plot(recon_losses, label="Reconstruction Loss (BCE)")
plt.plot(kld_losses, label="KL Divergence Loss")
plt.plot(total_losses, label="Total Loss (BCE + KLD)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE Training Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(recon_losses, label="Reconstruction Loss (BCE)")
plt.plot(kld_losses, label="KL Divergence Loss")
plt.plot(total_losses, label="Total Loss (BCE + KLD)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VAE Training Loss")
plt.legend()
plt.grid()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import torch

def visualize_latent_space(model, data_loader, num_samples=1000):
    model.eval()
    latent_vectors = []
    labels = []

    # Extract latent space (mu) for samples
    with torch.no_grad():
        for i, (images, direction_binary, _, _) in enumerate(data_loader):
            if i * len(images) > num_samples:  # Limit number of samples
                break
            images = images.view(-1, 64 * 64 * 3)  # Flatten
            mu, _ = model.encode(images)  # Encode into latent space
            latent_vectors.append(mu)
            labels.extend(direction_binary.numpy())  # Use binary labels for visualization

    # Stack latent vectors
    latent_vectors = torch.cat(latent_vectors).numpy()

    # Reduce dimensions (if latent_dim > 2)
    tsne = TSNE(n_components=2, random_state=42, perplexity=2)
    latent_2d = tsne.fit_transform(latent_vectors)

    # Plot the 2D latent space
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(latent_2d[:, 0], latent_2d[:, 1], c=labels, cmap="coolwarm", alpha=0.7)
    plt.colorbar(scatter, label="Direction Binary")
    plt.title("Latent Space Visualization (t-SNE)")
    plt.xlabel("Latent Dimension 1")
    plt.ylabel("Latent Dimension 2")
    plt.grid()
    plt.show()


In [ ]:
visualize_latent_space(model, val_loader, num_samples=10000)


In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_latent_features(model, data_loader, latent_dim_1=0, latent_dim_2=1, num_samples=1000):
    model.eval()
    latent_vectors = []
    labels = []

    # Extract latent space (mu) for samples
    with torch.no_grad():
        for i, (images, direction_binary, _, _) in enumerate(data_loader):
            if i * len(images) > num_samples:  # Limit the number of samples
                break
            images = images.view(-1, 64 * 64 * 3)  # Flatten the images
            mu, _ = model.encode(images)  # Encode to latent space
            latent_vectors.append(mu)
            labels.extend(direction_binary.numpy())  # Use direction binary for coloring

    # Stack latent vectors
    latent_vectors = torch.cat(latent_vectors).numpy()

    # Select specific latent dimensions
    dim1 = latent_vectors[:, latent_dim_1]
    dim2 = latent_vectors[:, latent_dim_2]

    # Plot the selected dimensions
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(dim1, dim2, c=labels, cmap="coolwarm", alpha=0.7)
    plt.colorbar(scatter, label="Direction Binary")
    plt.title(f"Latent Space: Dimension {latent_dim_1} vs Dimension {latent_dim_2}")
    plt.xlabel(f"Latent Dimension {latent_dim_1}")
    plt.ylabel(f"Latent Dimension {latent_dim_2}")
    plt.grid()
    plt.show()


In [ ]:
plot_latent_features(model, val_loader, latent_dim_1=0, latent_dim_2=1, num_samples=1000)


In [ ]:
def plot_latent_features(model, data_loader, latent_dim_1=0, latent_dim_2=1, num_samples=1000, label_type="direction_binary"):
    """
    Visualize two specific dimensions of the latent space.

    Parameters:
        model: The VAE model.
        data_loader: DataLoader to iterate through data.
        latent_dim_1: Index of the first latent dimension to plot.
        latent_dim_2: Index of the second latent dimension to plot.
        num_samples: Number of samples to use for visualization.
        label_type: The feature to use for coloring ('direction_binary', 'strength', etc.).
    """
    model.eval()
    latent_vectors = []
    labels = []

    with torch.no_grad():
        for i, (images, direction_binary, strength, segment_number) in enumerate(data_loader):
            if i * len(images) > num_samples:  # Limit the number of samples
                break
            images = images.view(-1, 64 * 64 * 3)  # Flatten images
            mu, _ = model.encode(images)  # Encode to latent space
            latent_vectors.append(mu)

            # Use selected label type for coloring
            if label_type == "direction_binary":
                labels.extend(direction_binary.numpy())
            elif label_type == "strength":
                labels.extend(strength.numpy())
            elif label_type == "segment_number":
                labels.extend(segment_number.numpy())
            else:
                raise ValueError(f"Unknown label_type: {label_type}")

    # Stack latent vectors
    latent_vectors = torch.cat(latent_vectors).numpy()

    # Select specific latent dimensions
    dim1 = latent_vectors[:, latent_dim_1]
    dim2 = latent_vectors[:, latent_dim_2]

    # Plot the selected dimensions
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(dim1, dim2, c=labels, cmap="coolwarm", alpha=0.7)
    plt.colorbar(scatter, label=label_type.capitalize())
    plt.title(f"Latent Space: Dimension {latent_dim_1} vs Dimension {latent_dim_2}")
    plt.xlabel(f"Latent Dimension {latent_dim_1}")
    plt.ylabel(f"Latent Dimension {latent_dim_2}")
    plt.grid()
    plt.show()


In [ ]:
plot_latent_features(model, val_loader, latent_dim_1=4, latent_dim_2=3, num_samples=1000, label_type="strength")


In [ ]:
# Generate synthetic images for the remaining training set segments
def generate_synthetic_images_for_training(model, train_loader, original_indices):
    model.eval()
    with torch.no_grad():
        for batch_idx, (images, _, _, segment_numbers) in enumerate(train_loader):  # Use train_loader
            print(f"Processing training batch {batch_idx + 1}...")

            # Flatten images to match VAE input
            images = images.view(-1, 64 * 64 * 3)
            recon_images, _, _ = model(images)
            recon_images = recon_images.view(-1, 3, 64, 64)  # Reshape to image dimensions

            # Save each reconstructed image with the correct index as in the original dataset
            for i in range(recon_images.size(0)):
                # Retrieve the correct segment index from `segment_numbers`
                segment_index = segment_numbers[i].item()  # Ensure segment index matches original dataset
                print(f"Saving synthetic image with segment index: {segment_index}")

                # Convert tensor to numpy, denormalize to [0, 255], and convert to uint8
                reconstructed_img = recon_images[i].permute(1, 2, 0).cpu().numpy() * 255.0
                reconstructed_img = reconstructed_img.astype(np.uint8)

                # Convert RGB to BGR for OpenCV compatibility
                reconstructed_img_bgr = cv2.cvtColor(reconstructed_img, cv2.COLOR_RGB2BGR)

                # Save the reconstructed image using the segment index
                save_path = os.path.join(output_reconstructed_dir, f"synthetic_mfm_segment_{segment_index}.png")
                cv2.imwrite(save_path, reconstructed_img_bgr)

In [ ]:
# Generate synthetic images for the remaining training set segments
generate_synthetic_images_for_training(model, train_loader, train_loader.dataset.indices)

In [ ]:
def verify_original_vs_synthetic(img_dir, synthetic_dir, indices_to_check):
    """
    Compare original and synthetic images side-by-side for given indices.
    
    Parameters:
        img_dir (str): Directory containing the original images.
        synthetic_dir (str): Directory containing the synthetic images.
        indices_to_check (list): List of indices to verify.
    """
    for index in indices_to_check:
        # Load the original image
        original_path = os.path.join(img_dir, f"mfm_segment_{index}.png")
        original_img = cv2.imread(original_path)
        original_img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

        # Load the synthetic image
        synthetic_path = os.path.join(synthetic_dir, f"synthetic_mfm_segment_{index}.png")
        synthetic_img = cv2.imread(synthetic_path)
        synthetic_img_rgb = cv2.cvtColor(synthetic_img, cv2.COLOR_BGR2RGB)

        # Plot original and synthetic images side-by-side
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(original_img_rgb)
        plt.title(f"Original Image (Segment {index})")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(synthetic_img_rgb)
        plt.title(f"Synthetic Image (Segment {index})")
        plt.axis('off')

        plt.show()

In [ ]:
# Directories for original and synthetic images
original_img_dir = "mfm_segments"
synthetic_img_dir = "synthetic_mfm_segments"

# List of indices to check
indices_to_check = [25, 33, 78, 89, 7, 81, 91]  # Sample indices; adjust as needed

# Run verification
verify_original_vs_synthetic(original_img_dir, synthetic_img_dir, indices_to_check)

In [ ]:
# Directory containing synthetic segments
mfm_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
segment_index = 10
segment_path = os.path.join(mfm_segments_dir, f"synthetic_mfm_segment_{segment_index}.png")

# Load and process the segment
segment = cv2.imread(segment_path, cv2.IMREAD_COLOR)
if segment is None:
    print(f"Segment {segment_index} could not be loaded.")
else:
    # Convert the segment to grayscale and create a binary mask
    segment_gray = cv2.cvtColor(segment, cv2.COLOR_BGR2GRAY)
    _, bit_mask = cv2.threshold(segment_gray, 10, 255, cv2.THRESH_BINARY)

    # Find contours and filter by area
    contours, _ = cv2.findContours(bit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_contour_area = 50  # Set minimum contour area threshold
    contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

    if contours:
        # Get bounding box and diagonal length for the largest contour
        x, y, w, h = cv2.boundingRect(contours[0])
        bit_diagonal_length = int(np.sqrt(w ** 2 + h ** 2))
        max_arrow_length = int(0.90 * bit_diagonal_length)
        min_arrow_length = int(0.55 * bit_diagonal_length)
        print(f"Bounding box (x={x}, y={y}, w={w}, h={h}), Diagonal length: {bit_diagonal_length}")

        # Fit an ellipse to find the orientation
        if len(contours[0]) >= 5:
            ellipse = cv2.fitEllipse(contours[0])
            center = (int(ellipse[0][0]), int(ellipse[0][1]))
            angle = ellipse[2]

            # Rotate the major axis by 90 degrees
            rotated_angle = angle + 90
            major_axis_dx = int(w * math.cos(math.radians(rotated_angle)))
            major_axis_dy = int(h * math.sin(math.radians(rotated_angle)))
            brightness_jump_threshold_percentage = 0.15  # Set threshold for sudden brightness jumps

            # Scan each half along the rotated major axis
            def scan_half_brightness(mask, center, major_axis_dx, major_axis_dy, half_name):
                pixels = np.column_stack(np.where(mask > 0))
                if pixels.size == 0:
                    return 0

                distances = (pixels[:, 0] - center[0]) * major_axis_dx + (pixels[:, 1] - center[1]) * major_axis_dy
                sorted_pixels = pixels[np.argsort(distances)]
                brightness_values = np.array([segment_gray[tuple(p)] for p in sorted_pixels], dtype=np.float32)
                brightness_gradients = np.abs(np.diff(brightness_values))
                brightness_range = brightness_values.max() - brightness_values.min()
                threshold = max(brightness_range * brightness_jump_threshold_percentage, 20)

                filtered_brightness = [brightness_values[0]]
                for i in range(1, len(brightness_values)):
                    if brightness_gradients[i - 1] < threshold:
                        filtered_brightness.append(brightness_values[i])

                avg_brightness = np.mean(filtered_brightness) if len(filtered_brightness) > 0 else 0
                print(f"{half_name} half adjusted brightness (filtered): {avg_brightness}")
                return avg_brightness

            # Calculate brightness for each half
            mask1, mask2 = np.zeros_like(bit_mask), np.zeros_like(bit_mask)
            for i in range(bit_mask.shape[0]):
                for j in range(bit_mask.shape[1]):
                    dx, dy = j - center[0], i - center[1]
                    projection = dx * major_axis_dx + dy * major_axis_dy
                    if projection > 0:
                        mask1[i, j] = bit_mask[i, j]
                    else:
                        mask2[i, j] = bit_mask[i, j]

            brightness1 = scan_half_brightness(mask1, center, major_axis_dx, major_axis_dy, "left")
            brightness2 = scan_half_brightness(mask2, center, major_axis_dx, major_axis_dy, "right")

            # Determine the brighter side and calculate arrow length
            is_side1_brighter = brightness1 > brightness2
            brightness_difference = abs(brightness1 - brightness2)
            normalized_strength = brightness_difference / 255.0
            arrow_length = max(min_arrow_length, min_arrow_length + int(normalized_strength * (max_arrow_length - min_arrow_length)))

            # Calculate arrow direction based on brightness
            angle_adjusted = rotated_angle if is_side1_brighter else rotated_angle + 180
            dx_brightest = int((arrow_length / 2) * math.cos(math.radians(angle_adjusted)))
            dy_brightest = int((arrow_length / 2) * math.sin(math.radians(angle_adjusted)))

            # Define and draw arrow
            start_point = (center[0] - dx_brightest, center[1] - dy_brightest)
            end_point = (center[0] + dx_brightest, center[1] + dy_brightest)
            segment_with_arrow = segment.copy()
            cv2.arrowedLine(segment_with_arrow, start_point, end_point, (105, 25, 255), 2, tipLength=0.2)

            # Display the segment with arrow
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(segment_with_arrow, cv2.COLOR_BGR2RGB))
            plt.title(f"Segment {segment_index} with Arrow Pointing to Brighter Side\nStrength: {arrow_length}")
            plt.axis('off')
            plt.show()

            print(f"Magnetic moment direction for segment {segment_index}: {'left' if is_side1_brighter else 'right'}")
            print(f"Magnetic moment strength (arrow length): {arrow_length}")
            print(f"Side 1 brightness (adjusted): {brightness1}")
            print(f"Side 2 brightness (adjusted): {brightness2}")
            print(f"Brightness difference (contrast): {brightness_difference}")

In [ ]:
# Define input and output directories
mfm_segments_dir = "{}/synthetic_mfm_segments".format(os.getcwd())
output_dir = "{}/synthetic_mfm_segments_with__updated_arrow_colors".format(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

# CSV file to save direction and strength of each segment
csv_path = os.path.join(output_dir, "synthetic_magnetic_moments.csv")
with open(csv_path, mode='w', newline='') as file:
    csv_writer = csv.writer(file)
    csv_writer.writerow(["Segment", "Direction", "Magnetic_Moment_Strength"])

    # Initialize counters for direction statistics
    left_arrow_count = 0
    right_arrow_count = 0
    net_magnetic_moment = 0
    min_arrow_length = float('inf')
    max_arrow_length = float('-inf')

    # Loop over each segment in the directory
    segment_files = sorted([f for f in os.listdir(mfm_segments_dir) if f.startswith("synthetic_mfm_segment_") and f.endswith(".png")])
    for segment_file in segment_files:
        # Extract segment index
        segment_index = int(segment_file.split('_')[-1].split('.')[0])
        segment_path = os.path.join(mfm_segments_dir, segment_file)

        # Load and process each segment
        segment = cv2.imread(segment_path, cv2.IMREAD_COLOR)
        if segment is None:
            print(f"Segment {segment_index} could not be loaded. Skipping...")
            continue

        # Convert the segment to grayscale and create a binary mask
        segment_gray = cv2.cvtColor(segment, cv2.COLOR_BGR2GRAY)
        _, bit_mask = cv2.threshold(segment_gray, 10, 255, cv2.THRESH_BINARY)

        # Find contours and filter by area to exclude small artifacts
        contours, _ = cv2.findContours(bit_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        min_contour_area = 50
        contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

        if not contours:
            print(f"No valid contours found for segment {segment_index}. Skipping...")
            continue

        # Get bounding box and diagonal length for the largest contour
        x, y, w, h = cv2.boundingRect(contours[0])
        bit_diagonal_length = int(np.sqrt(w ** 2 + h ** 2))
        max_arrow_length = int(0.90 * bit_diagonal_length)
        min_arrow_length = int(0.55 * bit_diagonal_length)

        # Fit an ellipse to find orientation and split segment along the major axis
        if len(contours[0]) >= 5:
            ellipse = cv2.fitEllipse(contours[0])
            center = (int(ellipse[0][0]), int(ellipse[0][1]))
            angle = ellipse[2]
            rotated_angle = angle + 90
            major_axis_dx = int(w * math.cos(math.radians(rotated_angle)))
            major_axis_dy = int(h * math.sin(math.radians(rotated_angle)))
            brightness_jump_threshold_percentage = 0.15
            fixed_threshold = 20

            # Function to calculate adjusted brightness for each half
            def scan_half_brightness(mask, center, major_axis_dx, major_axis_dy, half_name):
                pixels = np.column_stack(np.where(mask > 0))
                if pixels.size == 0:
                    return 0

                distances = (pixels[:, 0] - center[0]) * major_axis_dx + (pixels[:, 1] - center[1]) * major_axis_dy
                sorted_pixels = pixels[np.argsort(distances)]
                brightness_values = np.array([segment_gray[tuple(p)] for p in sorted_pixels], dtype=np.float32)

                # Filter brightness values for sudden jumps
                filtered_brightness = [brightness_values[0]]
                brightness_gradients = np.abs(np.diff(brightness_values))
                brightness_range = brightness_values.max() - brightness_values.min()
                threshold = max(brightness_range * brightness_jump_threshold_percentage, fixed_threshold)

                for i in range(1, len(brightness_values)):
                    if brightness_gradients[i - 1] < threshold:
                        filtered_brightness.append(brightness_values[i])

                avg_brightness = np.mean(filtered_brightness) if len(filtered_brightness) > 0 else 0
                return avg_brightness

            # Split mask into two halves along rotated major axis and calculate brightness
            mask1, mask2 = np.zeros_like(bit_mask), np.zeros_like(bit_mask)
            for i in range(bit_mask.shape[0]):
                for j in range(bit_mask.shape[1]):
                    dx, dy = j - center[0], i - center[1]
                    projection = dx * major_axis_dx + dy * major_axis_dy
                    if projection > 0:
                        mask1[i, j] = bit_mask[i, j]
                    else:
                        mask2[i, j] = bit_mask[i, j]

            brightness1 = scan_half_brightness(mask1, center, major_axis_dx, major_axis_dy, "Side 1")
            brightness2 = scan_half_brightness(mask2, center, major_axis_dx, major_axis_dy, "Side 2")

            # Determine the brighter side and calculate arrow length
            is_side1_brighter = brightness1 > brightness2
            brightness_difference = abs(brightness1 - brightness2)
            normalized_strength = brightness_difference / 255.0
            arrow_length = max(min_arrow_length, min_arrow_length + int(normalized_strength * (max_arrow_length - min_arrow_length)))

            # Calculate arrow direction
            angle_adjusted = rotated_angle if is_side1_brighter else rotated_angle + 180
            dx_brightest = int((arrow_length / 2) * math.cos(math.radians(angle_adjusted)))
            dy_brightest = int((arrow_length / 2) * math.sin(math.radians(angle_adjusted)))
            start_point = (center[0] - dx_brightest, center[1] - dy_brightest)
            end_point = (center[0] + dx_brightest, center[1] + dy_brightest)

            # Draw arrow and save
            segment_with_arrow = segment.copy()
            cv2.arrowedLine(segment_with_arrow, start_point, end_point, (105, 25, 255), 2, tipLength=0.2)
            output_path = os.path.join(output_dir, f"synthetic_mfm_segment_{segment_index}_with_arrow.png")
            cv2.imwrite(output_path, segment_with_arrow)

            # Direction is 1 for left, 0 for right
            direction_label = 1 if dx_brightest < 0 else 0
            if direction_label == 1:
                left_arrow_count += 1
                net_magnetic_moment -= arrow_length
            else:
                right_arrow_count += 1
                net_magnetic_moment += arrow_length

            # Save to CSV
            csv_writer.writerow([segment_file, direction_label, arrow_length])

            # Update arrow length range
            min_arrow_length = min(min_arrow_length, arrow_length)
            max_arrow_length = max(max_arrow_length, arrow_length)

            # Print for verification
            print(f"Segment {segment_index}: Direction: {'Left' if direction_label == 1 else 'Right'}, Strength: {arrow_length}")

    # Print summary statistics
    print("\nTotal arrows pointing right:", right_arrow_count)
    print("Total arrows pointing left:", left_arrow_count)
    print("Net magnetic moment (based on arrow lengths):", net_magnetic_moment)
    print("Arrow length range:", min_arrow_length, "to", max_arrow_length)
    overall_magnetic_moment = "Right" if net_magnetic_moment > 0 else "Left"
    print("The overall magnetic moment direction is:", overall_magnetic_moment)

In [ ]:


# Directories for synthetic segments and output
synthetic_segments_dir = "{}/synthetic_mfm_segments_with__updated_arrow_colors".format(os.getcwd())
original_segments_dir = "{}/bit_visualization_segments".format(os.getcwd())
output_dir = "{}/reconstructed_image_updated_arrow_color".format(os.getcwd())
os.makedirs(output_dir, exist_ok=True)

# Path to the original "Bits Segmentation Visualization" image
segmentation_visualization_path = "bit_segmentation_visualization.png"

# Load the original segmentation visualization to get the original dimensions
segmentation_visualization = cv2.imread(segmentation_visualization_path)
if segmentation_visualization is None:
    print("Segmentation visualization image could not be loaded.")
else:
    # Create a blank canvas for the reconstructed image
    reassembled_image = np.zeros_like(segmentation_visualization)

    # Convert to grayscale and find connected components in the segmentation visualization
    segmentation_gray = cv2.cvtColor(segmentation_visualization, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(segmentation_gray, 1, 255, cv2.THRESH_BINARY)
    num_labels, labels_im = cv2.connectedComponents(binary_mask)

    # Step 1: Loop over each segment label in the mask
    for label in range(1, num_labels):
        # Paths for synthetic and original segments
        synthetic_segment_path = os.path.join(synthetic_segments_dir, f"synthetic_mfm_segment_{label}_with_arrow.png")
        original_segment_path = os.path.join(original_segments_dir, f"segment_{label}.png")

        # Load both synthetic and original segments
        synthetic_segment = cv2.imread(synthetic_segment_path, cv2.IMREAD_COLOR)
        original_segment = cv2.imread(original_segment_path, cv2.IMREAD_COLOR)

        if synthetic_segment is None:
            print(f"Synthetic Segment {label} could not be loaded. Skipping...")
            continue
        if original_segment is None:
            print(f"Original Segment {label} could not be loaded. Skipping...")
            continue

        # Remove the background of the synthetic segment
        synthetic_gray = cv2.cvtColor(synthetic_segment, cv2.COLOR_BGR2GRAY)
        _, synthetic_mask = cv2.threshold(synthetic_gray, 10, 255, cv2.THRESH_BINARY)
        x_s, y_s, w_s, h_s = cv2.boundingRect(synthetic_mask)
        cropped_synthetic = synthetic_segment[y_s:y_s + h_s, x_s:x_s + w_s]

        # Find the bounding box of the segment in the original mask
        mask = (labels_im == label).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print(f"No contours found for segment {label}. Skipping...")
            continue

        # Get the bounding box of the current segment area in the mask
        x, y, w, h = cv2.boundingRect(contours[0])

        # Resize the cropped synthetic segment to match the bounding box of the original segment
        synthetic_resized = cv2.resize(cropped_synthetic, (w, h))

        # Overlay the synthetic segment onto the reassembled image
        reassembled_image[y:y + h, x:x + w] = synthetic_resized

        # # Step 2: Write the segment number on top of the bit
        # label_position = (x + w // 2, y + h // 2)
        # cv2.putText(reassembled_image, str(label), label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 255, 0), 1, cv2.LINE_AA)

    # Step 3: Display the reassembled image
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(reassembled_image, cv2.COLOR_BGR2RGB))
    plt.title("Reconstructed MFM Image from Synthetic Segments with Arrows")
    plt.axis('off')
    plt.show()


In [ ]:
# # Directories for synthetic segments and output
# synthetic_segments_dir = "{}/synthetic_mfm_segments_with_arrows".format(os.getcwd())
# original_segments_dir = "{}/bit_visualization_segments".format(os.getcwd())
# output_dir = "{}/reconstructed_image".format(os.getcwd())
# os.makedirs(output_dir, exist_ok=True)

# # Path to the original "Bits Segmentation Visualization" image
# segmentation_visualization_path = "bit_segmentation_visualization.png"

# # Load the original segmentation visualization to get the original dimensions
# segmentation_visualization = cv2.imread(segmentation_visualization_path)
# if segmentation_visualization is None:
#     print("Segmentation visualization image could not be loaded.")
# else:
#     # Create a blank canvas for the reconstructed image
#     reassembled_image = np.zeros_like(segmentation_visualization)

#     # Convert to grayscale and find connected components in the segmentation visualization
#     segmentation_gray = cv2.cvtColor(segmentation_visualization, cv2.COLOR_BGR2GRAY)
#     _, binary_mask = cv2.threshold(segmentation_gray, 1, 255, cv2.THRESH_BINARY)
#     num_labels, labels_im = cv2.connectedComponents(binary_mask)

#     # Step 1: Loop over each segment label in the mask
#     for label in range(1, num_labels):
#         # Paths for synthetic and original segments
#         synthetic_segment_path = os.path.join(synthetic_segments_dir, f"synthetic_mfm_segment_{label}_with_arrow.png")
#         original_segment_path = os.path.join(original_segments_dir, f"segment_{label}.png")

#         # Load both synthetic and original segments
#         synthetic_segment = cv2.imread(synthetic_segment_path, cv2.IMREAD_COLOR)
#         original_segment = cv2.imread(original_segment_path, cv2.IMREAD_COLOR)

#         if synthetic_segment is None:
#             print(f"Synthetic Segment {label} could not be loaded. Skipping...")
#             continue
#         if original_segment is None:
#             print(f"Original Segment {label} could not be loaded. Skipping...")
#             continue

#         # Remove the background of the synthetic segment
#         synthetic_gray = cv2.cvtColor(synthetic_segment, cv2.COLOR_BGR2GRAY)
#         _, synthetic_mask = cv2.threshold(synthetic_gray, 10, 255, cv2.THRESH_BINARY)
#         x_s, y_s, w_s, h_s = cv2.boundingRect(synthetic_mask)
#         cropped_synthetic = synthetic_segment[y_s:y_s + h_s, x_s:x_s + w_s]

#         # Find the bounding box of the segment in the original mask
#         mask = (labels_im == label).astype(np.uint8) * 255
#         contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#         if not contours:
#             print(f"No contours found for segment {label}. Skipping...")
#             continue

#         # Get the bounding box of the current segment area in the mask
#         x, y, w, h = cv2.boundingRect(contours[0])

#         # Resize the cropped synthetic segment to match the bounding box of the original segment
#         synthetic_resized = cv2.resize(cropped_synthetic, (w, h))

#         # Overlay the synthetic segment onto the reassembled image
#         reassembled_image[y:y + h, x:x + w] = synthetic_resized
        
#         # Step 2: Write the segment number on top of the bit
#         label_position = (x + w // 2, y + h // 2)
#         cv2.putText(reassembled_image, str(label), label_position, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)


#     # Step 2: Display the reassembled image
#     plt.figure(figsize=(10, 10))
#     plt.imshow(cv2.cvtColor(reassembled_image, cv2.COLOR_BGR2RGB))
#     plt.title("Reconstructed MFM Image from Synthetic Segments with Arrows")
#     plt.axis('off')
#     plt.show()
    

In [ ]:
reassembled_image_path =  "{}/reconstructed_synthetic_mfm_image_with_directional_updated_arrows_paper_fig.png".format(os.getcwd())
cv2.imwrite(reassembled_image_path, reassembled_image)
print(f"Reconstructed MFM image with labels saved at: {reassembled_image_path}")


# reassembled_image_path = os.path.join(os.getcwd(), "reconstructed_synthetic_mfm_image_with_directional_arrows_paper_fig.png")
# plt.savefig(reassembled_image_path, bbox_inches='tight', dpi=300)  # Save with high resolution
# print(f"Image saved to {reassembled_image_path}")

# Error mapping - original & VAE Image

In [ ]:

# Paths to the reconstructed synthetic and original MFM images
synthetic_image_path = "{}/reconstructed_synthetic_mfm_image_with_directional_arrows.png".format(os.getcwd())
original_image_path = "{}/reconstructed_mfm_image_with_directional_arrows.png".format(os.getcwd())

# Load images and convert them to grayscale
synthetic_image = cv2.imread(synthetic_image_path, cv2.IMREAD_GRAYSCALE)
original_image = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)

if synthetic_image is None or original_image is None:
    print("One or both images could not be loaded.")
else:
    # Ensure images are the same size
    if synthetic_image.shape != original_image.shape:
        synthetic_image = cv2.resize(synthetic_image, (original_image.shape[1], original_image.shape[0]))

    # Binarize the images to create classes (threshold can be adjusted)
    _, synthetic_binary = cv2.threshold(synthetic_image, 127, 1, cv2.THRESH_BINARY)
    _, original_binary = cv2.threshold(original_image, 127, 1, cv2.THRESH_BINARY)

    # Flatten the images for pixel-wise comparison
    synthetic_flat = synthetic_binary.flatten()
    original_flat = original_binary.flatten()

    # Calculate the confusion matrix
    cm = confusion_matrix(original_flat, synthetic_flat, labels=[0, 1])

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Background', 'Magnetic Element'], yticklabels=['Background', 'Magnetic Element'])
    plt.xlabel("Synthetic Image Classes")
    plt.ylabel("Original Image Classes")
    plt.title("Confusion Matrix: Original vs. Synthetic Reconstructed MFM Image")
    plt.show()


In [ ]:
# Paths to the reconstructed images
original_reconstructed_path = "reconstructed_mfm_image_with_directional_arrows.png"
synthetic_reconstructed_path = "reconstructed_synthetic_mfm_image_with_directional_arrows.png"


In [ ]:
# Load the images in grayscale
original_reconstructed = cv2.imread(original_reconstructed_path, cv2.IMREAD_GRAYSCALE)
synthetic_reconstructed = cv2.imread(synthetic_reconstructed_path, cv2.IMREAD_GRAYSCALE)

# Ensure that both images are loaded
if original_reconstructed is None or synthetic_reconstructed is None:
    print("One of the reconstructed images could not be loaded.")
    exit()

# Resize images to the same dimensions if necessary
if original_reconstructed.shape != synthetic_reconstructed.shape:
    target_size = (min(original_reconstructed.shape[1], synthetic_reconstructed.shape[1]),
                   min(original_reconstructed.shape[0], synthetic_reconstructed.shape[0]))
    original_reconstructed = cv2.resize(original_reconstructed, target_size)
    synthetic_reconstructed = cv2.resize(synthetic_reconstructed, target_size)

# Calculate the absolute difference
difference_map = cv2.absdiff(original_reconstructed, synthetic_reconstructed)

# Normalize the difference map for visualization
difference_map_normalized = cv2.normalize(difference_map, None, 0, 255, cv2.NORM_MINMAX)

# Plot the difference map as a heatmap with a reduced height colorbar
plt.figure(figsize=(8, 8))
heatmap = plt.imshow(difference_map_normalized, cmap='hot')
cbar = plt.colorbar(heatmap, fraction=0.04, pad=0.04)  # Adjust 'fraction' for height
cbar.set_label("Pixel Accuracy")
plt.title("Pixel Difference Heatmap Between Reconstructed Original and Synthetic MFM Images")
plt.axis("off")
plt.show()

In [ ]:
original_reconstructed = cv2.imread(original_reconstructed_path, cv2.IMREAD_GRAYSCALE)
synthetic_reconstructed = cv2.imread(synthetic_reconstructed_path, cv2.IMREAD_GRAYSCALE)

# Resize images if necessary to ensure they're the same size
if original_reconstructed.shape != synthetic_reconstructed.shape:
    target_size = (min(original_reconstructed.shape[1], synthetic_reconstructed.shape[1]),
                   min(original_reconstructed.shape[0], synthetic_reconstructed.shape[0]))
    original_reconstructed = cv2.resize(original_reconstructed, target_size)
    synthetic_reconstructed = cv2.resize(synthetic_reconstructed, target_size)

# Calculate the Mean Squared Error (MSE) map
mse_map = (original_reconstructed.astype("float") - synthetic_reconstructed.astype("float")) ** 2

# Compute the Structural Similarity Index (SSIM) map
ssim_score, ssim_map = ssim(original_reconstructed, synthetic_reconstructed, full=True)
ssim_map = (ssim_map * 255).astype("uint8")  # Convert SSIM map to 8-bit for visualization

# Plot MSE and SSIM maps
plt.figure(figsize=(12, 6))

# MSE heatmap
plt.subplot(1, 2, 1)
plt.imshow(mse_map, cmap='hot')
plt.colorbar(label="MSE Intensity")
plt.title("Mean Squared Error (MSE) Map")

# SSIM heatmap
plt.subplot(1, 2, 2)
plt.imshow(ssim_map, cmap='hot')
plt.colorbar(label="SSIM Intensity")
plt.title(f"Structural Similarity Index (SSIM) Map\nAverage SSIM Score: {ssim_score:.3f}")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_vae_architecture():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axis("off")

    # Input
    ax.text(0.1, 0.8, "Input\n(64x64x3)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightblue"))

    # Encoder
    ax.text(0.3, 0.8, "Flatten\n(12288)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightgreen"))
    ax.text(0.5, 0.8, "FC1\n(12288→400)\nReLU", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="orange"))
    ax.text(0.7, 0.85, "Latent Mean\n(400→60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="pink"))
    ax.text(0.7, 0.75, "Latent LogVar\n(400→60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="pink"))
    ax.text(0.85, 0.8, "Reparameterization\nz = μ + σ * ε\n(60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="yellow"))

    # Decoder
    ax.text(1.0, 0.8, "FC3\n(60→400)\nReLU", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="orange"))
    ax.text(1.2, 0.8, "FC4\n(400→12288)\nSigmoid", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightgreen"))
    ax.text(1.4, 0.8, "Output\n(64x64x3)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightblue"))

    # Arrows
    ax.annotate("", xy=(0.2, 0.8), xytext=(0.1, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.4, 0.8), xytext=(0.3, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.6, 0.8), xytext=(0.5, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.68, 0.85), xytext=(0.6, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.68, 0.75), xytext=(0.6, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.8, 0.8), xytext=(0.7, 0.85), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.8, 0.8), xytext=(0.7, 0.75), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.9, 0.8), xytext=(0.85, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(1.1, 0.8), xytext=(1.0, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(1.3, 0.8), xytext=(1.2, 0.8), arrowprops=dict(arrowstyle="->"))

    # Show and save
    plt.tight_layout()
    #plt.savefig("vae_architecture_matplotlib.png")
    plt.show()

# Call the function
plot_vae_architecture()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_vae_custom():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axis("off")

    # Input
    ax.text(0.1, 0.8, "Input\n(64x64x3)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightblue"))

    # Flatten
    ax.text(0.3, 0.8, "Flatten\n(12288)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightgreen"))

    # Encoder Dense Layer
    ax.text(0.5, 0.8, "Encoder Dense\n(12288 → 400)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="orange"))

    # Latent State
    ax.text(0.7, 0.85, "Latent Mean\n(400 → 60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="pink"))
    ax.text(0.7, 0.75, "Latent LogVar\n(400 → 60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="pink"))
    ax.text(0.85, 0.8, "Reparameterization\nz = μ + σ * ε\n(60)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="yellow"))

    # Decoder Dense Layers
    ax.text(1.0, 0.8, "Decoder Dense\n(60 → 400)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="orange"))
    ax.text(1.2, 0.8, "Decoder Output\n(400 → 12288)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightgreen"))

    # Output
    ax.text(1.4, 0.8, "Output\n(64x64x3)", fontsize=12, ha="center", bbox=dict(boxstyle="round", facecolor="lightblue"))

    # Arrows
    ax.annotate("", xy=(0.2, 0.8), xytext=(0.1, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.4, 0.8), xytext=(0.3, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.6, 0.8), xytext=(0.5, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.68, 0.85), xytext=(0.6, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.68, 0.75), xytext=(0.6, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.8, 0.8), xytext=(0.7, 0.85), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.8, 0.8), xytext=(0.7, 0.75), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(0.9, 0.8), xytext=(0.85, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(1.1, 0.8), xytext=(1.0, 0.8), arrowprops=dict(arrowstyle="->"))
    ax.annotate("", xy=(1.3, 0.8), xytext=(1.2, 0.8), arrowprops=dict(arrowstyle="->"))

    # Save and show
    plt.tight_layout()
    # plt.savefig("vae_custom_architecture.png")
    plt.show()

plot_vae_custom()
